In [10]:
import os
import shutil
import re
from pathlib import Path
import pandas as pd
from Constants import Const
import warnings
from pathlib import Path

### Old 486 patients

In [11]:
folderpath = Path("/Volumes/Siyuan SSD/DICOM_20260421_old/Processed")
missing_csv = []
missing_roi_values = []
checked_patient_folders = []
csv_read_errors = []

required_rois = {"GTV_p", "GTV_n"}

for subfolder in sorted(folderpath.iterdir()):
    if not subfolder.is_dir():
        continue

    for patient_folder in sorted(subfolder.iterdir()):
        if not patient_folder.is_dir():
            continue

        checked_patient_folders.append(patient_folder)

        rtstruct_folder = patient_folder / "RTSTRUCT_ABAS"
        csv_path = rtstruct_folder / "CT_centroid.csv"

        if not csv_path.exists():
            missing_csv.append(patient_folder)
            continue

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            csv_read_errors.append({
                "patient_folder": str(patient_folder),
                "csv_path": str(csv_path),
                "error": str(e)
            })
            continue

        if "ROI" not in df.columns:
            missing_roi_values.append({
                "patient_folder": str(patient_folder),
                "csv_path": str(csv_path),
                "missing": sorted(required_rois),
                "reason": "ROI column not found"
            })
            continue

        roi_values = set(df["ROI"].dropna().astype(str).str.strip())
        missing = required_rois - roi_values

        if missing:
            missing_roi_values.append({
                "patient_folder": str(patient_folder),
                "csv_path": str(csv_path),
                "missing": sorted(missing),
                "reason": "Required ROI value missing"
            })

print("=" * 80)
print("Summary")
print("=" * 80)
print(f"Total patient folders checked: {len(checked_patient_folders)}")
print(f"Missing CT_centroid.csv count: {len(missing_csv)}")
print(f"Files with missing ROI values count: {len(missing_roi_values)}")
print(f"CSV read error count: {len(csv_read_errors)}")

print("\n" + "=" * 80)
print("Patient folders missing CT_centroid.csv")
print("=" * 80)

if missing_csv:
    for p in missing_csv:
        print(p)
else:
    print("None")

print("\n" + "=" * 80)
print("Existing CT_centroid.csv files with missing GTV_p or GTV_n")
print("=" * 80)

if missing_roi_values:
    for item in missing_roi_values:
        print(f"Patient folder: {item['patient_folder']}")
        print(f"CSV path: {item['csv_path']}")
        print(f"Missing ROI values: {item['missing']}")
        print(f"Reason: {item['reason']}")
        print("-" * 80)
else:
    print("None")

print("\n" + "=" * 80)
print("CSV read errors")
print("=" * 80)

if csv_read_errors:
    for item in csv_read_errors:
        print(f"Patient folder: {item['patient_folder']}")
        print(f"CSV path: {item['csv_path']}")
        print(f"Error: {item['error']}")
        print("-" * 80)
else:
    print("None")

Summary
Total patient folders checked: 486
Missing CT_centroid.csv count: 58
Files with missing ROI values count: 160
CSV read error count: 0

Patient folders missing CT_centroid.csv
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_1/105025355210
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_1/139123597330
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_1/140881477949
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_10/131129643906
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_10/217361683229
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_10/555557827139
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_2/154270148296
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_2/156320095591
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_2/160908779974
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_2/161120525775
/Volumes/Siyuan SSD/DICOM_20260421_old/Processed/Batch_2/162338411679
/Volumes/Siyuan SSD/DICOM_20260421_old/Proce

In [12]:
excluded_gtv_missing_id = ['105025355210', '139123597330', '140881477949', '154270148296', '156320095591', '160908779974', '161120525775', '162338411679', '167800999244', '175393242095', '175858867340', '176268482619', '184285931319', '195672393332', '197965012329', '203362125714', '205876857238', '214335244635', '226926195577', '230439881947', '235393299667', '240295756907', '247377995066', '276772052044', '277089814380', '277431880233', '280489084994', '288910275196', '289499607380', '290806098388', '293974098976', '303572115007', '310026611461', '319274969366', '319292636320', '320500592855', '332157140963', '344254882250', '344968892802', '369407392063', '381696011115', '462557850342', '484006968728', '511051809968', '707454175961', '709317957738', '753408672849', '764961562475', '875433292830', '884514874829', '899667420050', '910319911165', '931251318367', '962607992104', '987810235978', '131129643906']
excluded_organ_missing_id = ['217361683229', '555557827139']
huge_id = ['646769528112']

all_excluded_ids = excluded_gtv_missing_id + excluded_organ_missing_id + huge_id
len(all_excluded_ids)

59

In [14]:
excluded_set = set(str(x).strip() for x in all_excluded_ids)

missing_csv_ids = [p.name for p in missing_csv]

missing_csv_ids_clean = [
    x.replace("STIEFEL_", "").strip()
    for x in missing_csv_ids
]

remaining_missing_csv_ids = [
    x for x in missing_csv_ids_clean
    if x not in excluded_set
]

print("=" * 80)
print("Missing CT_centroid.csv IDs after excluding excluded_gtv_missing_id")
print("=" * 80)
print(f"Original missing CT_centroid.csv count: {len(missing_csv_ids_clean)}")
print(f"Excluded ID count: {len(excluded_set)}")
print(f"Remaining count: {len(remaining_missing_csv_ids)}")

print("\nRemaining IDs:")
for x in remaining_missing_csv_ids:
    print(x)

Missing CT_centroid.csv IDs after excluding excluded_gtv_missing_id
Original missing CT_centroid.csv count: 58
Excluded ID count: 59
Remaining count: 0

Remaining IDs:


In [15]:
# File names expected inside each patient's RTSTRUCT folder
CENTROID_FILE = "CT_centroid.csv"
DISTANCE_FILE = "CT_distances.csv"
RTSTRUCT_FOLDER = "RTSTRUCT_ABAS"


# Output directory under data/
output_dir = "../data/CAMPRT_Centroids_20260421_old"
os.makedirs(output_dir, exist_ok=True)

copied = []
missing = []

# Iterate: folderpath -> subfolders -> patient folders -> RTSTRUCT
for subfolder in sorted(os.listdir(folderpath)):
    subfolder_path = os.path.join(folderpath, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for patient_id in sorted(os.listdir(subfolder_path)):
        patient_path = os.path.join(subfolder_path, patient_id)
        if not os.path.isdir(patient_path):
            continue

        rtstruct_path = os.path.join(patient_path, RTSTRUCT_FOLDER)
        if not os.path.isdir(rtstruct_path):
            missing.append((patient_id, "RTSTRUCT folder not found"))
            continue

        # Create patient output directory
        patient_out_dir = os.path.join(output_dir, patient_id)
        os.makedirs(patient_out_dir, exist_ok=True)

        for filename in [CENTROID_FILE, DISTANCE_FILE]:
            src = os.path.join(rtstruct_path, filename)
            dst = os.path.join(patient_out_dir, filename)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                copied.append((patient_id, filename))
            else:
                missing.append((patient_id, filename))

print(f"Copied {len(copied)} files for {len(set(p for p, _ in copied))} patients.")
if missing:
    print(f"\nMissing ({len(missing)}):")
    for patient_id, item in missing:
        print(f"  Patient {patient_id}: {item}")

Copied 856 files for 428 patients.

Missing (114):
  Patient 105025355210: CT_centroid.csv
  Patient 105025355210: CT_distances.csv
  Patient 139123597330: CT_centroid.csv
  Patient 139123597330: CT_distances.csv
  Patient 140881477949: CT_centroid.csv
  Patient 140881477949: CT_distances.csv
  Patient 131129643906: CT_centroid.csv
  Patient 131129643906: CT_distances.csv
  Patient 217361683229: RTSTRUCT folder not found
  Patient 555557827139: RTSTRUCT folder not found
  Patient 154270148296: CT_centroid.csv
  Patient 154270148296: CT_distances.csv
  Patient 156320095591: CT_centroid.csv
  Patient 156320095591: CT_distances.csv
  Patient 160908779974: CT_centroid.csv
  Patient 160908779974: CT_distances.csv
  Patient 161120525775: CT_centroid.csv
  Patient 161120525775: CT_distances.csv
  Patient 162338411679: CT_centroid.csv
  Patient 162338411679: CT_distances.csv
  Patient 167800999244: CT_centroid.csv
  Patient 167800999244: CT_distances.csv
  Patient 175393242095: CT_centroid.csv

In [16]:
organ_rename_dict = {
    "cricoid": "Cricoid_cartilage",
    "cricopharyngeus": "Cricopharyngeal_Muscle",
    "esophagus_u": "Esophagus",
    "oral_cavity": "Extended_Oral_Cavity",
    "musc_geniogloss": "Genioglossus_M",
    "hardpalate": "Hard_Palate",
    "bone_hyoid": "Hyoid_bone",
    "musc_constrict_i": "IPC",
    "lips_lower": "Lower_Lip",
    "lips_upper": "Upper_Lip",
    "musc_constrict_m": "MPC",
    "musc_mgh_complex": "Mylogeniohyoid_M",
    "musc_mghcomplex": "Mylogeniohyoid_M",
    "palate_soft": "Soft_Palate",
    "musc_constrict_s": "SPC",
    "spinalcord_cerv": "Spinal_Cord",
    "larynx_sg": "Supraglottic_Larynx",
    "cartlg_thyroid": "Thyroid_cartilage",
    "brachial_plex_r": "Rt_Brachial_Plexus",
    "brachial_plex_l": "Lt_Brachial_Plexus",
    "pterygoid_lat_r": "Rt_Lateral_Pterygoid_M",
    "pterygoid_lat_l": "Lt_Lateral_Pterygoid_M",
    "musc_masseter_r": "Rt_Masseter_M",
    "musc_masseter_l": "Lt_Masseter_M",
    "bone_mastoid_r": "Rt_Mastoid",
    "bone_mastoid_l": "Lt_Mastoid",
    "pterygoid_med_r": "Rt_Medial_Pterygoid_M",
    "pterygoid_med_l": "Lt_Medial_Pterygoid_M",
    "parotid_r": "Rt_Parotid_Gland",
    "parotid_l": "Lt_Parotid_Gland",
    "musc_sclmast_r": "Rt_Sternocleidomastoid_M",
    "musc_sclmast_l": "Lt_Sternocleidomastoid_M",
    "glnd_submand_r": "Rt_Submandibular_Gland",
    "glnd_submand_l": "Lt_Submandibular_Gland",
    "musc_digastric_ra": "Rt_Ant_Digastric_M",
    "musc_digastric_la": "Lt_Ant_Digastric_M",
    "musc_digastric_rp": "Rt_Post_Digastric_M",
    "musc_digastric_lp": "Lt_Post_Digastric_M",
    "Esophagus_U": "Esophagus",
    "esophagus": "Esophagus",
    "Esophagus_up": "Esophagus",
    "Esophagus_Up": "Esophagus",
    "cool down eso": "Esophagus",
    "fs shaper esoph": "Esophagus",
    "Partial Esophagus": "Esophagus",
    "fsPushEso": "Esophagus",
    "z_eso push": "Esophagus",
    "fs shaper eso": "Esophagus",
    "fsEsophMax28": "Esophagus",
    "fsEsophMax30": "Esophagus",
    "FS esoph opt": "Esophagus",
    "eso sub": "Esophagus",
    "fs esoph hot": "Esophagus",
    "lower esophagus": "Esophagus",
    "SpinalCord": "Spinal_Cord",
    "SpinalCord_PRV05": "Spinal_Cord",
    "SpinalCord_Cerv": "Spinal_Cord",
    "SpinalCord_PRV5": "Spinal_Cord",
    "SpinalCordPRV_05": "Spinal_Cord",
    "SpinalCord_05": "Spinal_Cord",
    "SpinalCord_PRV02": "Spinal_Cord",
    "SpinalCord_03": "Spinal_Cord",
    "BrachialPlex_L": "Lt_Brachial_Plexus",
    "fsL_BrachPlex exp": "Lt_Brachial_Plexus",
    "BrachialPlex_left": "Lt_Brachial_Plexus",
    "Lt Plexus hot": "Lt_Brachial_Plexus",
    "Brachial_Plex_L": "Lt_Brachial_Plexus",
    "Brach Plex_L": "Lt_Brachial_Plexus",
    "Lt brachial plexus 58": "Lt_Brachial_Plexus",
    "Lt BrachialPlex slice 81 and below": "Lt_Brachial_Plexus",
    "BrachialPlex_L_lower": "Lt_Brachial_Plexus",
    "BrachialPlex_L Inferior": "Lt_Brachial_Plexus",
    "Brachial_Plex_R": "Rt_Brachial_Plexus",
    "BrachialPlex_R": "Rt_Brachial_Plexus",
    "BrachialPlex_right": "Rt_Brachial_Plexus",
    "Brachial_R Expanded 2mm": "Rt_Brachial_Plexus",
    "Rplexusexp1mm": "Rt_Brachial_Plexus",
    "Rt plexus hot": "Rt_Brachial_Plexus",
    "Brach Plex_R": "Rt_Brachial_Plexus",
    "rt brachial plexus 58": "Rt_Brachial_Plexus",
    "Rt_BrachialPlex slice 81 and below": "Rt_Brachial_Plexus",
    "BrachialPlex R": "Rt_Brachial_Plexus",
    "BrachialPlex_R lower": "Rt_Brachial_Plexus",
    "cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricopharyngeus & Arytenoids": "Cricopharyngeal_Muscle",
    "Cricopharyngeus & Arytenoid": "Cricopharyngeal_Muscle",
    "Arytenoid & Cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricoid": "Cricoid_cartilage",
    "Musc_Constrict_I": "IPC",
    "Musc_Constrict_M": "MPC",
    "Musc_Constrict_S": "SPC",
    "Brainstem": "Brainstem",
    "Brainstem_PRV05": "Brainstem",
    "BrainStem": "Brainstem",
    "BrainStem_PRV5": "Brainstem",
    "Brainstem_PRV5": "Brainstem",
    "BrainStem_05": "Brainstem",
    "Brainstem_03": "Brainstem",
    "Brainstem Expanded 5mm": "Brainstem",
    "Brainstem expanded 5mm": "Brainstem",
    "BrainStem_03": "Brainstem",
    "cool down brainstem": "Brainstem",
    "cool down cord_brainstem": "Brainstem",
    "Brainstem_PRV03": "Brainstem",
    "Brainstem_PRV02": "Brainstem",
    "Brainstem +0.5": "Brainstem",
    "Larynx": "Larynx",
    "larynx sub": "Larynx",
    "Larynx_SG": "Larynx",
    "fslarynxhot": "Larynx",
    "Larynx Proper": "Larynx",
    "Larynx proper": "Larynx",
    "FS Larynx Opt": "Larynx",
    "fsLarynxStrip2": "Larynx",
    "fslarynxexp": "Larynx",
    "fsLarynx_plan": "Larynx",
    "larynx": "Larynx",
    "fsCool larynx": "Larynx",
    "Larynx sub": "Larynx",
    "post larynx": "Larynx",
    "pLarynx": "Larynx",
    "dgLarynx_42": "Larynx",
    "xLarynx": "Larynx",
    "z_larynx push": "Larynx",
    "fs_larynx hard": "Larynx",
    "fs larynx av": "Larynx",
    "fs Larynx av": "Larynx",
    "fs shaper larynx": "Larynx",
    "fs larynx shaper": "Larynx",
    "fs larynx shaper2": "Larynx",
    "fs ant larynx": "Larynx",
    "z_hardlarynx": "Larynx",
    "fs larynx horn": "Larynx",
    "FS larynx opt": "Larynx",
    "fs larynx push": "Larynx",
    "fs Avoivd Eso Larynx and Thryoid": "Larynx",
    "FS Larynx opt": "Larynx",
    "fs larynx original": "Larynx",
    "fs no30_larynx": "Larynx",
    "Cartlg_Thyroid": "Thyroid_cartilage",
    "Glnd_Thyroid_L": "Thyroid_cartilage",
    "Glnd_Thyroid_R": "Thyroid_cartilage",
    "Thyroid": "Thyroid_cartilage",
    "thyroid": "Thyroid_cartilage",
    "fs thyroid push": "Thyroid_cartilage",
    "Glnd_Thyroid": "Thyroid_cartilage",
    "fs_thyroid push": "Thyroid_cartilage",
    "thyroid sub": "Thyroid_cartilage",
    "Thryoid": "Thyroid_cartilage",
    "Musc_Sclmast_R": "Rt_Sternocleidomastoid_M",
    "Bone_Mastoid_R": "Rt_Mastoid",
    "fs mastoid push": "Rt_Mastoid",
    "fs r mastoid push": "Rt_Mastoid",
    "Parotid_R": "Rt_Parotid_Gland",
    "fsParotid_R_Sub": "Rt_Parotid_Gland",
    "fs rt parotid push": "Rt_Parotid_Gland",
    "fsRparotidhot": "Rt_Parotid_Gland",
    "fsParotid_R_Sub_1": "Rt_Parotid_Gland",
    "fsRparotid_out": "Rt_Parotid_Gland",
    "fsCool rt parotid": "Rt_Parotid_Gland",
    "fs rt parotid low push": "Rt_Parotid_Gland",
    "R Parotid push": "Rt_Parotid_Gland",
    "Rt Parotid top": "Rt_Parotid_Gland",
    "fsRt Parotid push": "Rt_Parotid_Gland",
    "fsRtParotid10Gypush": "Rt_Parotid_Gland",
    "Rt Parotid push": "Rt_Parotid_Gland",
    "fs rt parotid sub": "Rt_Parotid_Gland",
    "fs parotid_Rsub": "Rt_Parotid_Gland",
    "Sub_Parotid_R": "Rt_Parotid_Gland",
    "rt parotid push": "Rt_Parotid_Gland",
    "z_parotidR_in70": "Rt_Parotid_Gland",
    "z_parotid_R_push": "Rt_Parotid_Gland",
    "R parotid push": "Rt_Parotid_Gland",
    "fs_rparotid top": "Rt_Parotid_Gland",
    "fs parotids push": "Rt_Parotid_Gland",
    "Parotid_Critical_R": "Rt_Parotid_Gland",
    "fs Parotid Top": "Rt_Parotid_Gland",
    "Pterygoid_Med_R": "Rt_Medial_Pterygoid_M",
    "Pterygoid_Lat_R": "Rt_Lateral_Pterygoid_M",
    "Musc_Masseter_R": "Rt_Masseter_M",
    "Musc_Sclmast_L": "Lt_Sternocleidomastoid_M",
    "Bone_Mastoid_L": "Lt_Mastoid",
    "Parotid_L": "Lt_Parotid_Gland",
    "fsParotid_L_Sub": "Lt_Parotid_Gland",
    "fs lt parotid push": "Lt_Parotid_Gland",
    "fsLparotidhot": "Lt_Parotid_Gland",
    "Lt parotid push2": "Lt_Parotid_Gland",
    "fs Lt parotid push": "Lt_Parotid_Gland",
    "fsParotid_L_Sub_1": "Lt_Parotid_Gland",
    "fs lt parotid low push": "Lt_Parotid_Gland",
    "L Parotid push": "Lt_Parotid_Gland",
    "Lt Parotid top": "Lt_Parotid_Gland",
    "fsParotid_L_Sub1": "Lt_Parotid_Gland",
    "fsParotid_L_Sub2": "Lt_Parotid_Gland",
    "fs Lt Parotid Push": "Lt_Parotid_Gland",
    "fa lt parotid push": "Lt_Parotid_Gland",
    "fs lt parotid": "Lt_Parotid_Gland",
    "lt parotid push": "Lt_Parotid_Gland",
    "fs lt parotid sub": "Lt_Parotid_Gland",
    "fs lt parotid push 2": "Lt_Parotid_Gland",
    "fs parotid_Lsub": "Lt_Parotid_Gland",
    "Sub_Parotid_L": "Lt_Parotid_Gland",
    "L parotid sub": "Lt_Parotid_Gland",
    "Lt Parotid push": "Lt_Parotid_Gland",
    "Parotid_Critical_L": "Lt_Parotid_Gland",
    "New Lt Parotid Sub": "Lt_Parotid_Gland",
    "Glnd_Submand_L": "Lt_Submandibular_Gland",
    "Submandibular_L": "Lt_Submandibular_Gland",
    "left submandibular gland": "Lt_Submandibular_Gland",
    "Submandibular Gland_L": "Lt_Submandibular_Gland",
    "Submandubular_L": "Lt_Submandibular_Gland",
    "l submandibular": "Lt_Submandibular_Gland",
    "z_submand L_push": "Lt_Submandibular_Gland",
    "DNU Glnd_Submand_L": "Lt_Submandibular_Gland",
    "glnd_submand_l": "Lt_Submandibular_Gland",
    "Pterygoid_Med_L": "Lt_Medial_Pterygoid_M",
    "Pterygoid_Lat_L": "Lt_Lateral_Pterygoid_M",
    "Musc_Masseter_L": "Lt_Masseter_M",
    "larynx_sg": "Supraglottic_Larynx",
    "glnd_submand_r": "Rt_Submandibular_Gland",
    "Glnd_Submand_R": "Rt_Submandibular_Gland",
    "Submandibular_R": "Rt_Submandibular_Gland",
    "z_submand push_R": "Rt_Submandibular_Gland",
    "DNU Glnd_Submand_R": "Rt_Submandibular_Gland",
    "Bone_Hyoid": "Hyoid_bone",
    "Palate_Soft": "Soft_Palate",
    "5700 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "6300 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "7000 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "Musc_Geniogloss": "Genioglossus_M",
    "Tongue": "Tongue",
    "tongue": "Tongue",
    "4500 (fff Base of Tongue JPR)": "Tongue",
    "5700 (fff Base of Tongue JPR)": "Tongue",
    "6300 (fff Base of Tongue JPR)": "Tongue",
    "6600 (fff Base of Tongue JPR)": "Tongue",
    "7000 (fff Base of Tongue JPR)": "Tongue",
    "fs tongue push": "Tongue",
    "Musc_Digastric_RA": "Rt_Ant_Digastric_M",
    "Musc_Digastric_RP": "Rt_Ant_Digastric_M",
    "Musc_Digastric_LA": "Lt_Ant_Digastric_M",
    "Musc_Digastric_LP": "Lt_Ant_Digastric_M",
    "Musc_MGHComplex": "Mylogeniohyoid_M",
    "Cavity_Oral": "Extended_Oral_Cavity",
    "Oral_Cavity": "Extended_Oral_Cavity",
    "OralCavity": "Extended_Oral_Cavity",
    "CavityOral": "Extended_Oral_Cavity",
    "Oral Cavity": "Extended_Oral_Cavity",
    "OralCavity_Original": "Extended_Oral_Cavity",
    "fsOralCavity": "Extended_Oral_Cavity",
    "oral cavity": "Extended_Oral_Cavity",
    "zCavity_Oral (1)": "Extended_Oral_Cavity",
    "xCavity_Oral": "Extended_Oral_Cavity",
    "Cavity_Oral push": "Extended_Oral_Cavity",
    "z_Cavity_Oral_push": "Extended_Oral_Cavity",
    "Bone_Mandible": "Mandible",
    "Mandible": "Mandible",
    "fsCool mandible": "Mandible",
    "fs7300mandible": "Mandible",
    "z_mandible IN": "Mandible",
    "fs_73mandible": "Mandible",
    "fs_hotmandible": "Mandible",
    "fsMandiblePush": "Mandible",
    "fs mandible hot": "Mandible",
    "fs mandible": "Mandible",
    "zMaxmandible": "Mandible",
    "mandible avd": "Mandible",
    "fsMandible sub": "Mandible",
    "Hardpalate": "Hard_Palate",
    "Lips_Lower": "Lower_Lip",
    "Lips_Upper": "Upper_Lip",
}

In [17]:
len(Const.organ_list)

40

In [18]:
target_dir = Path("../data/CAMPRT_Centroids_20260421_old").resolve()

excluded = {str(x) for x in all_excluded_ids}
removed, missing = [], []

for pid in excluded:
    folder = target_dir / pid
    if not folder.is_dir():
        missing.append(pid)
        continue
    shutil.rmtree(folder)
    removed.append(pid)

print(f"removed {len(removed)} folders:", removed)
if missing:
    print(f"not found (skip): {len(missing)}", missing)

removed 57 folders: ['290806098388', '247377995066', '293974098976', '303572115007', '884514874829', '226926195577', '240295756907', '319274969366', '140881477949', '195672393332', '289499607380', '381696011115', '131129643906', '161120525775', '319292636320', '310026611461', '175393242095', '205876857238', '230439881947', '139123597330', '167800999244', '646769528112', '332157140963', '344968892802', '235393299667', '484006968728', '910319911165', '160908779974', '288910275196', '156320095591', '709317957738', '753408672849', '764961562475', '462557850342', '344254882250', '511051809968', '175858867340', '707454175961', '184285931319', '899667420050', '320500592855', '277089814380', '369407392063', '203362125714', '162338411679', '280489084994', '214335244635', '875433292830', '197965012329', '931251318367', '987810235978', '105025355210', '276772052044', '277431880233', '962607992104', '154270148296', '176268482619']
not found (skip): 2 ['217361683229', '555557827139']


In [19]:
# --- DVH → CT_centroid (requires organ_rename_dict + canonical_roi from cells above) ---
DATA = Path("../data").resolve()
PATIENT_ID_XLSX_PATHS = [
    DATA / "Anonymized_DVH_Values____to_be_shared" / "PatientIDs_merged_Anonymized_INCLUDING_STIEFEL_IDs.xlsx",
    DATA
    / "Anonymized_DVH_Values____to_be_shared-for__Batch_10"
    / "PatientIDs_Anonymized_INCLUDING_STIEFEL_IDs.xlsx",
]
OAR_DVH_DIR = DATA / "DVH_20260421_old"
GTV_DVH_DIR = DATA / "DVH_20260421_old_only_gtv"
DOSE_CENTROID_ROOT = Path("../data/CAMPRT_Centroids_20260421_old").resolve()
ORGAN_SET = set(Const.organ_list)
# Case-insensitive lookup tables: organ_rename_dict mixes lower/mixed-case keys,
# canonical_roi only does .lower()-based lookup so mixed-case entries never hit.
_ORGAN_RENAME_LC = {str(k).strip().lower(): v for k, v in organ_rename_dict.items()}
_ORGAN_LC = {o.lower(): o for o in ORGAN_SET}


def _patient_id_str(v):
    if v is None or pd.isna(v):
        return None
    if isinstance(v, bool):
        return None
    try:
        return str(int(v))
    except (ValueError, TypeError):
        pass
    s = str(v).strip()
    if not s:
        return None
    try:
        return str(int(float(s)))
    except ValueError:
        return s


def _resolve_oar(name):
    """Case-insensitive resolver. Returns Const.organ_list name or None."""
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    key = str(name).strip().lower()
    if not key:
        return None
    if key == "larynx_sg":
        return "Supraglottic_Larynx"
    if key in _ORGAN_RENAME_LC:
        label = _ORGAN_RENAME_LC[key]
        return label if label in ORGAN_SET else None
    if key in _ORGAN_LC:
        return _ORGAN_LC[key]
    return None


def _centroid_canon_key(name):
    """Map CT_centroid ROI value → canonical key for merge.

    Keeps GTV_p/GTV_n as-is; OARs go through _resolve_oar; everything else → None
    so prune step drops it and dose merge stays NaN rather than misalign.
    """
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    s = str(name).strip()
    if s in {"GTV_p", "GTV_n"}:
        return s
    return _resolve_oar(s)


def load_stiefel_to_mrn(paths: list[Path]) -> dict:
    """Map STIEFEL_ID → MRN (Anonymized_ID) using the two patient-id xlsx files."""
    mapping: dict[str, str] = {}
    for p in paths:
        df = pd.read_excel(p)
        anon_col = next(
            (c for c in df.columns if "anonymized_id" in c.lower()),
            None,
        )
        stiefel_col = next((c for c in df.columns if c.strip().upper() == "STIEFEL_ID"), None)
        if anon_col is None or stiefel_col is None:
            raise ValueError(f"Missing Anonymized_ID/STIEFEL_ID columns in {p}")
        for _, row in df.iterrows():
            sid = row[stiefel_col]
            mrn = _patient_id_str(row[anon_col])
            if sid is None or (isinstance(sid, float) and pd.isna(sid)) or mrn is None:
                continue
            mapping[str(sid).strip()] = mrn
    return mapping


def _load_dvh_batch_table(folder: Path, glob_pattern: str, stiefel_to_mrn: dict) -> pd.DataFrame:
    """Concat all DVH batch csvs in `folder`, attach MRN via STIEFEL_ID mapping.

    Returns df with columns: STIEFEL_ID, Structure (raw), MRN, Volume, mean, minGy, maxGy.
    """
    files = sorted(folder.glob(glob_pattern))
    if not files:
        return pd.DataFrame(columns=["STIEFEL_ID", "Structure", "MRN", "Volume", "mean", "minGy", "maxGy"])
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    id_col = next(
        (c for c in df.columns if c.lstrip("﻿").strip().lower() == "id"),
        df.columns[0],
    )
    df = df.rename(columns={id_col: "STIEFEL_ID"})
    df["STIEFEL_ID"] = df["STIEFEL_ID"].astype(str).str.strip()
    df["MRN"] = df["STIEFEL_ID"].map(stiefel_to_mrn)
    df = df.dropna(subset=["MRN"])
    df["Structure"] = df["Structure"].astype(str).str.strip()
    return df[["STIEFEL_ID", "Structure", "MRN", "Volume", "mean", "minGy", "maxGy"]]


def load_oar_dose_table(folder: Path, stiefel_to_mrn: dict) -> pd.DataFrame:
    """OAR DVH rows resolved to Const.organ_list names; one row per (MRN, organ)."""
    df = _load_dvh_batch_table(folder, "DVH_all_ABAS_only_batch_*.csv", stiefel_to_mrn)
    df["match_roi"] = df["Structure"].map(_resolve_oar)
    df = df.dropna(subset=["match_roi"])
    return df[["MRN", "match_roi", "Volume", "mean", "minGy", "maxGy"]].drop_duplicates(
        subset=["MRN", "match_roi"], keep="last"
    )


def load_gtv_dose_table(folder: Path, stiefel_to_mrn: dict) -> pd.DataFrame:
    """GTV-only DVH rows; one row per (MRN, GTV_p|GTV_n)."""
    df = _load_dvh_batch_table(folder, "DVH_RTSTRUCT_GTV_only_batch_*.csv", stiefel_to_mrn)
    df = df[df["Structure"].isin({"GTV_p", "GTV_n"})].rename(columns={"Structure": "match_roi"})
    return df[["MRN", "match_roi", "Volume", "mean", "minGy", "maxGy"]].drop_duplicates(
        subset=["MRN", "match_roi"], keep="last"
    )


def prune_centroid_and_distance_files(target_dir: Path, *, dry_run: bool = False):
    """Drop ROIs not in Const.organ_list ∪ {GTV_p, GTV_n} from CT_centroid.csv and CT_distances.csv,
    rewriting both with canonical ROI names."""
    for folder in sorted(target_dir.iterdir()):
        if not folder.is_dir():
            continue
        cpath = folder / "CT_centroid.csv"
        if cpath.is_file():
            cdf = pd.read_csv(cpath)
            if "ROI" in cdf.columns:
                cdf["ROI"] = cdf["ROI"].map(_centroid_canon_key)
                cdf = cdf.dropna(subset=["ROI"]).drop_duplicates(subset=["ROI"], keep="first")
                if not dry_run:
                    cdf.to_csv(cpath, index=False)
        dpath = folder / "CT_distances.csv"
        if dpath.is_file():
            ddf = pd.read_csv(dpath)
            if {"Reference ROI", "Target ROI"}.issubset(ddf.columns):
                ddf["Reference ROI"] = ddf["Reference ROI"].map(_centroid_canon_key)
                ddf["Target ROI"] = ddf["Target ROI"].map(_centroid_canon_key)
                ddf = ddf.dropna(subset=["Reference ROI", "Target ROI"]).drop_duplicates(
                    subset=["Reference ROI", "Target ROI"], keep="first"
                )
                if not dry_run:
                    ddf.to_csv(dpath, index=False)


def write_dvh_doses_to_centroid_files(
    target_dir: Path,
    oar_df: pd.DataFrame,
    gtv_df: pd.DataFrame,
    *,
    dry_run: bool = False,
):
    combined = pd.concat([oar_df, gtv_df], ignore_index=True)
    by_mrn = {mrn: g for mrn, g in combined.groupby("MRN")}
    for folder in sorted(target_dir.iterdir()):
        if not folder.is_dir():
            continue
        mrn = folder.name.strip()
        cpath = folder / "CT_centroid.csv"
        if not cpath.is_file():
            continue
        cdf = pd.read_csv(cpath)
        if "ROI" not in cdf.columns:
            continue
        g = by_mrn.get(mrn)
        if g is None:
            mt = pd.DataFrame(
                columns=["match_roi", "Min Value", "Mean Value", "Max Value", "Structure Volume"]
            )
        else:
            mt = g.rename(
                columns={
                    "minGy": "Min Value",
                    "mean": "Mean Value",
                    "maxGy": "Max Value",
                    "Volume": "Structure Volume",
                }
            )[["match_roi", "Min Value", "Mean Value", "Max Value", "Structure Volume"]]
            mt = mt.drop_duplicates(subset=["match_roi"], keep="first")
        cdf = cdf.drop(
            columns=["Min Value", "Mean Value", "Max Value", "Structure Volume"],
            errors="ignore",
        )
        out = cdf.merge(mt, how="left", left_on="ROI", right_on="match_roi").drop(
            columns=["match_roi"], errors="ignore"
        )
        for c in ("Min Value", "Mean Value", "Max Value", "Structure Volume"):
            if c not in out.columns:
                out[c] = pd.NA
        if not dry_run:
            out.to_csv(cpath, index=False)


stiefel_to_mrn = load_stiefel_to_mrn(PATIENT_ID_XLSX_PATHS)
oar_dose_table = load_oar_dose_table(OAR_DVH_DIR, stiefel_to_mrn)
gtv_dose_table = load_gtv_dose_table(GTV_DVH_DIR, stiefel_to_mrn)
prune_centroid_and_distance_files(DOSE_CENTROID_ROOT, dry_run=False)
write_dvh_doses_to_centroid_files(
    DOSE_CENTROID_ROOT,
    oar_dose_table,
    gtv_dose_table,
    dry_run=False,
)


In [20]:
target_dir = Path("../data/CAMPRT_Centroids_20260421_old").resolve()
GTV_LABELS = frozenset({"GTV_p", "GTV_n"})

organ_set = set(Const.organ_list) | set(GTV_LABELS)


def canonical_roi(name: str):
    """Map CSV ROI label → Const.organ_list naming; return None if empty."""
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    s = str(name).strip()
    if not s:
        return None
    key = s.lower()
    if key in organ_rename_dict:
        return organ_rename_dict[key]
    for o in organ_set:
        if o.lower() == key:
            return o
    return s


def process_patient_folder(folder: Path, dry_run: bool = False) -> dict:
    out = {"folder": str(folder), "centroid": None, "distances": None}
    cpath = folder / "CT_centroid.csv"
    dpath = folder / "CT_distances.csv"

    if cpath.is_file():
        cdf = pd.read_csv(cpath)
        if "ROI" not in cdf.columns:
            warnings.warn(f"{cpath}: no ROI column")
        else:
            cdf["ROI"] = cdf["ROI"].map(canonical_roi)
            cdf = cdf.dropna(subset=["ROI"])
            cdf = cdf[cdf["ROI"].isin(organ_set)].drop_duplicates(subset=["ROI"])
            if not dry_run:
                cdf.to_csv(cpath, index=False)
            out["centroid"] = len(cdf)
    else:
        out["centroid"] = "missing"

    if dpath.is_file():
        ddf = pd.read_csv(dpath)
        need = ["Reference ROI", "Target ROI"]
        if not all(c in ddf.columns for c in need):
            warnings.warn(f"{dpath}: missing {need}")
        else:
            ddf["Reference ROI"] = ddf["Reference ROI"].map(canonical_roi)
            ddf["Target ROI"] = ddf["Target ROI"].map(canonical_roi)
            ddf = ddf.dropna(subset=need)
            mask = ddf["Reference ROI"].isin(organ_set) & ddf["Target ROI"].isin(
                organ_set
            )
            ddf = ddf.loc[mask]
            if not dry_run:
                ddf.to_csv(dpath, index=False)
            out["distances"] = len(ddf)
    else:
        out["distances"] = "missing"

    return out


summaries = []
for sub in sorted(target_dir.iterdir()):
    if sub.is_dir():
        summaries.append(process_patient_folder(sub, dry_run=False))

summary_df = pd.DataFrame(summaries)
print(summary_df.head(10))
print(
    "totals:",
    summary_df["centroid"].apply(lambda x: x if isinstance(x, int) else 0).sum(),
    "centroid rows (approx sum across patients),",
    summary_df["distances"].apply(lambda x: x if isinstance(x, int) else 0).sum(),
    "distance rows",
)

                                              folder  centroid  distances
0  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
1  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        41        828
2  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
3  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
4  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        41        828
5  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        41        828
6  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
7  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
8  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        42        869
9  /Users/siyuanzhao/Documents/GitHub/QubbedDataA...        41        828
totals: 17742 centroid rows (approx sum across patients), 363807 distance rows


### 319 patients processing

In [24]:
folderpath = "/Volumes/Siyuan SSD/DICOM_20260421_new/Processed"

In [25]:
# File names expected inside each patient's RTSTRUCT folder
CENTROID_FILE = "CT_centroid.csv"
DISTANCE_FILE = "CT_distances.csv"
RTSTRUCT_FOLDER = "RTSTRUCT"


# Output directory under data/
output_dir = "../data/CAMPRT_Centroids_20260421_new"
os.makedirs(output_dir, exist_ok=True)

copied = []
missing = []

# Iterate: folderpath -> subfolders -> patient folders -> RTSTRUCT
for subfolder in sorted(os.listdir(folderpath)):
    subfolder_path = os.path.join(folderpath, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for patient_id in sorted(os.listdir(subfolder_path)):
        patient_path = os.path.join(subfolder_path, patient_id)
        if not os.path.isdir(patient_path):
            continue

        rtstruct_path = os.path.join(patient_path, RTSTRUCT_FOLDER)
        if not os.path.isdir(rtstruct_path):
            missing.append((patient_id, "RTSTRUCT folder not found"))
            continue

        # Create patient output directory
        patient_out_dir = os.path.join(output_dir, patient_id)
        os.makedirs(patient_out_dir, exist_ok=True)

        for filename in [CENTROID_FILE, DISTANCE_FILE]:
            src = os.path.join(rtstruct_path, filename)
            dst = os.path.join(patient_out_dir, filename)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                copied.append((patient_id, filename))
            else:
                missing.append((patient_id, filename))

print(f"Copied {len(copied)} files for {len(set(p for p, _ in copied))} patients.")
if missing:
    print(f"\nMissing ({len(missing)}):")
    for patient_id, item in missing:
        print(f"  Patient {patient_id}: {item}")

Copied 638 files for 319 patients.


In [36]:
# ============================================================
# ROI selection helpers
# ============================================================

# Non-target name prefixes that should never be considered as GTV/CTV/PTV
# candidates: moat rings (avoidance) and Eclipse field-shaping helpers (fs*).
# These appear in CAMPRT-style centroid files but are not tumour targets.
_NON_TARGET_RE = re.compile(r'^\s*(?:moat|fs)', re.IGNORECASE)


def _is_target_candidate(name):
    """False for clearly non-target ROIs (moat_*, fs*)."""
    return _NON_TARGET_RE.match(name) is None


def _get_prefix_rois(names, prefix):
    """Return ROI names that contain the given prefix (case-insensitive)."""
    return [n for n in names if prefix in n.lower()]


def _is_primary_pattern(name, prefix):
    """
    True if `name` looks like a primary-tumour ROI for the given prefix.

    Matches either an explicit primary marker AFTER the prefix
        GTVp, GTV_p, GTV-P, GTV P, GTV primary, GTVp_5600, GTV_P_5600
    or a 'p' marker BEFORE the prefix
        pGTV, pCTV_6600, p_CTV, pPTV53
    The bare prefix itself ('GTV', 'CTV', 'PTV') also counts.
    """
    lower = name.lower().strip()
    if lower == prefix:
        return True
    after = rf'{prefix}[_\-\s]*(?:p(?:[^a-z]|$)|primary)'
    before = rf'(?:^|[^a-z])p[_\-\s]*{prefix}'
    return bool(re.search(after, lower) or re.search(before, lower))


def _is_nodal_pattern(name, prefix):
    """
    True if `name` looks like a nodal ROI for the given prefix.

    Matches an 'n' marker AFTER the prefix (GTVn, GTV-N, GTV_N1, GTV node)
    or BEFORE the prefix (nCTV, n_PTV).
    """
    lower = name.lower()
    after = rf'{prefix}[_\-\s]*n(?:[rl\d]|ode[s]?|[_\-\s]|$)'
    before = rf'(?:^|[^a-z])n[_\-\s]*{prefix}'
    return bool(re.search(after, lower) or re.search(before, lower))


def _dose_score(name):
    """
    Return the highest dose hint embedded in an ROI name (in cGy), or -1.

    Recognises both 4-digit cGy (CTV_6600 -> 6600) and 2-digit Gy shorthand
    (pPTV53 -> 5300). Used to break ties between dose-encoded targets such
    as CTV_6600 / CTV_6000 / CTV_5300, where the highest dose level is the
    primary tumour target in head-and-neck CAMPRT data.
    """
    best = -1
    for m in re.finditer(r'(\d+)', name):
        n = int(m.group(1))
        if 1000 <= n <= 9999:
            best = max(best, n)
        elif 30 <= n <= 99:
            best = max(best, n * 100)
    return best


def _highest_dose_target(subset, prefix):
    """Pick the non-nodal ROI in `subset` with the highest dose hint."""
    non_nodal = [n for n in subset if not _is_nodal_pattern(n, prefix)]
    with_dose = [(n, _dose_score(n)) for n in non_nodal]
    with_dose = [(n, d) for n, d in with_dose if d > 0]
    if not with_dose:
        return None
    with_dose.sort(key=lambda x: (-x[1], x[0]))
    return with_dose[0][0]


def find_primary_roi(matched_rois):
    """
    Select the primary tumour ROI (GTV_p) from a patient's matched ROI list.

    Priority order: GTV > CTV > PTV. Within each prefix:
      1. Exact base name ('GTV', 'CTV', 'PTV')
      2. Explicit primary marker (GTVp, GTV_P, GTV primary, pGTV, pCTV_6600, pPTV53)
      3. Highest dose-encoded non-nodal target (CTV_6600 over CTV_5300)
    Falls back to the first non-nodal candidate, then the first ROI.
    Non-target ROIs (moat_*, fs*) are filtered out first.
    """
    candidates = [n for n in matched_rois if _is_target_candidate(n)]

    for prefix in ["gtv", "ctv", "ptv"]:
        subset = _get_prefix_rois(candidates, prefix)
        if not subset:
            continue
        exact = [n for n in subset if n.lower().strip() == prefix]
        if exact:
            return exact[0]
        primary = [n for n in subset if _is_primary_pattern(n, prefix)]
        if primary:
            return primary[0]
        dose_pick = _highest_dose_target(subset, prefix)
        if dose_pick is not None:
            return dose_pick

    non_nodal = [
        n for n in candidates
        if not any(_is_nodal_pattern(n, p) for p in ["gtv", "ctv", "ptv"])
    ]
    if non_nodal:
        return non_nodal[0]
    if candidates:
        return candidates[0]
    return matched_rois[0] if matched_rois else None


def find_nodal_roi(matched_rois):
    """
    Select the nodal tumour ROI (GTV_n) from a patient's matched ROI list.
    Priority order: GTV > CTV > PTV. Returns None if no nodal ROI is found.
    Non-target ROIs (moat_*, fs*) are filtered out first.
    """
    candidates = [n for n in matched_rois if _is_target_candidate(n)]
    for prefix in ["gtv", "ctv", "ptv"]:
        subset = _get_prefix_rois(candidates, prefix)
        nodal = [n for n in subset if _is_nodal_pattern(n, prefix)]
        if nodal:
            return nodal[0]
    return None


def has_real_primary(matched_rois):
    """
    True if the matched ROIs contain a recognisable primary-tumour target
    (explicit p marker OR a dose-encoded non-nodal CTV/PTV/GTV).
    """
    candidates = [n for n in matched_rois if _is_target_candidate(n)]
    for prefix in ["gtv", "ctv", "ptv"]:
        for n in _get_prefix_rois(candidates, prefix):
            if _is_primary_pattern(n, prefix):
                return True
    for prefix in ["gtv", "ctv", "ptv"]:
        subset = _get_prefix_rois(candidates, prefix)
        if _highest_dose_target(subset, prefix) is not None:
            return True
    return False


# ============================================================
# Configuration
# ============================================================

ROOT_DIR = "../data/CAMPRT_Centroids_20260421_new"
CENTROID_FILE = "CT_centroid.csv"
DISTANCE_FILE = "CT_distances.csv"
# Same CSV shape as tumor_roi_selection_20260421_old.csv: batch, patient_id, GTV_p, GTV_n.
# Use e.g. BATCH_NAME_FOR_CSV = "Batch_1" to match that file's batch column.
BATCH_NAME_FOR_CSV = os.path.basename(ROOT_DIR)
TUMOR_ROI_SELECTION_CSV = "../data/tumor_roi_selection_20260421_new.csv"

# Candidate column names that may hold ROI names in CT_centroid.csv
ROI_COLUMN_CANDIDATES = ["ROI", "roi", "Roi", "name", "Name", "NAME",
                         "structure", "Structure", "STRUCTURE",
                         "organ", "Organ", "ORGAN"]


def extract_roi_names(csv_path):
    """
    Read a CT_centroid.csv file and return a list of ROI names.
    Tries common column names first; falls back to the first non-numeric column.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  [ERROR] Could not read {csv_path}: {e}")
        return []

    if df.empty:
        return []

    # Try known column names
    for col in ROI_COLUMN_CANDIDATES:
        if col in df.columns:
            return df[col].dropna().astype(str).tolist()

    # Fallback: first column that is not numeric
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            return df[col].dropna().astype(str).tolist()

    # Final fallback: just use the first column
    return df.iloc[:, 0].dropna().astype(str).tolist()


# ============================================================
# Scan patient folders
# ============================================================

if not os.path.isdir(ROOT_DIR):
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

patient_folders = sorted(
    name for name in os.listdir(ROOT_DIR)
    if os.path.isdir(os.path.join(ROOT_DIR, name))
)

patients_with_roi = []
missing_centroid_file = []

for patient in patient_folders:
    centroid_path = os.path.join(ROOT_DIR, patient, CENTROID_FILE)
    if not os.path.isfile(centroid_path):
        missing_centroid_file.append(patient)
        continue
    roi_names = extract_roi_names(centroid_path)
    patients_with_roi.append({
        "batch": BATCH_NAME_FOR_CSV,
        "patient": patient,
        "matched_rois": roi_names,
    })


# ============================================================
# Tumor ROI selection and reporting
# ============================================================

print("Tumor ROI Selection")
print("=" * 100)

no_gtv_p_patients = []
no_gtv_n_patients = []
selection_rows = []


def _roi_csv_cell(val):
    return "" if val is None else val


for item in patients_with_roi:
    batch = item["batch"]
    pid = item["patient"]
    rois = item["matched_rois"]

    if not rois:
        print(f"{batch}/{pid} | (no ROIs found in CT_centroid.csv)")
        no_gtv_p_patients.append(pid)
        no_gtv_n_patients.append(pid)
        selection_rows.append(
            {"batch": batch, "patient_id": pid, "GTV_p": "", "GTV_n": ""}
        )
        continue

    gtv_p = find_primary_roi(rois)
    gtv_n = find_nodal_roi(rois)

    if not has_real_primary(rois):
        no_gtv_p_patients.append(pid)
        gtv_p_display = f"{repr(gtv_p)} (fallback)" if gtv_p is not None else "None"
        print(
            f"{batch}/{pid:30s} | "
            f"GTV_p: {gtv_p_display:50s} | "
            f"GTV_n: {repr(gtv_n) if gtv_n is not None else 'None'}"
        )
        continue

    if gtv_n is None:
        no_gtv_n_patients.append(pid)

    selection_rows.append(
        {
            "batch": batch,
            "patient_id": pid,
            "GTV_p": _roi_csv_cell(gtv_p),
            "GTV_n": _roi_csv_cell(gtv_n),
        }
    )

    print(
        f"{batch}/{pid:30s} | "
        f"GTV_p: {repr(gtv_p):50s} | "
        f"GTV_n: {repr(gtv_n) if gtv_n is not None else 'None'}"
    )

# Folders without CT_centroid.csv: empty tumor columns (one combined table)
for pid in missing_centroid_file:
    selection_rows.append(
        {
            "batch": BATCH_NAME_FOR_CSV,
            "patient_id": pid,
            "GTV_p": "",
            "GTV_n": "",
        }
    )

selection_rows.sort(key=lambda r: (str(r["patient_id"]),))


# ============================================================
# Summary
# ============================================================

total = len(patients_with_roi)
found_primary = total - len(no_gtv_p_patients)
found_nodal = total - len(no_gtv_n_patients)

print()
print("Summary")
print("-" * 100)
print(f"Root directory          : {ROOT_DIR}")
print(f"Patient folders scanned : {len(patient_folders)}")
print(f"Folders with centroid   : {total}")
print(f"Missing centroid file   : {len(missing_centroid_file)}")
print()
print(f"GTV_p found  : {found_primary}/{total}")
print(f"GTV_p missing: {len(no_gtv_p_patients)}/{total}")
print(f"GTV_n found  : {found_nodal}/{total}")
print(f"GTV_n missing: {len(no_gtv_n_patients)}/{total}")

if no_gtv_p_patients:
    print()
    print("Patients without a real GTV_p match:")
    for p in no_gtv_p_patients:
        print(f"  - {p}")

if missing_centroid_file:
    print()
    print(f"Patients missing {CENTROID_FILE}:")
    for p in missing_centroid_file:
        print(f"  - {p}")

sel_df = pd.DataFrame(
    selection_rows, columns=["batch", "patient_id", "GTV_p", "GTV_n"]
)
os.makedirs(os.path.dirname(TUMOR_ROI_SELECTION_CSV) or ".", exist_ok=True)
sel_df.to_csv(TUMOR_ROI_SELECTION_CSV, index=False)
print()
print(
    "Wrote tumor ROI mapping:",
    os.path.abspath(TUMOR_ROI_SELECTION_CSV),
    f"({len(sel_df)} rows; columns: batch, patient_id, GTV_p, GTV_n)",
)
print("\nPreview (first 5 rows):")
print(sel_df.head().to_string(index=False))

Tumor ROI Selection
CAMPRT_Centroids_20260421_new/101417584648                   | GTV_p: 'A_Carotid_Int_L' (fallback)                       | GTV_n: None
CAMPRT_Centroids_20260421_new/102209659698                   | GTV_p: 'GTVp'                                             | GTV_n: 'GTVn'
CAMPRT_Centroids_20260421_new/102231587299                   | GTV_p: 'GTV_primary'                                      | GTV_n: 'GTV_nodes1'
CAMPRT_Centroids_20260421_new/102934140275                   | GTV_p: 'pCTV_6600'                                        | GTV_n: None
CAMPRT_Centroids_20260421_new/103512138945                   | GTV_p: 'pCTV_6600'                                        | GTV_n: 'GTV_Node_1'
CAMPRT_Centroids_20260421_new/103895810595                   | GTV_p: 'pCTV_6000'                                        | GTV_n: None
CAMPRT_Centroids_20260421_new/105559195097                   | GTV_p: 'pCTV_6600'                                        | GTV_n: 'GTV_Node'
CAMPRT_Cent

In [39]:
import os
import glob
import pandas as pd
from collections import defaultdict

from Constants import Const


# ============================================================
# Configuration
# ============================================================

ROOT_DIR = "../data/CAMPRT_Centroids_20260421_new"
CENTROID_FILE = "CT_centroid.csv"
DISTANCE_FILE = "CT_distances.csv"

ANON_MAPPING_XLSX = "../data/merged_sent_including_Stiefel_IDs_Anonymized.xlsx"
ANON_COL = "Anonymized"
ID_COL = "ID"

DVH_DIR = "../data/DVH_20260421_new"
TUMOR_ROI_SELECTION_CSV = "../data/tumor_roi_selection_20260421_new.csv"

# Column names in the CT_centroid.csv / CT_distances.csv
CENTROID_ROI_COL = "ROI"
DISTANCE_REF_COL = "Reference ROI"
DISTANCE_TGT_COL = "Target ROI"

# Column names in DVH csv files
DVH_ID_COL = "id"
DVH_ROI_COL = "Structure"
DVH_MEAN_COL = "mean"
DVH_MIN_COL = "minGy"
DVH_MAX_COL = "maxGy"
DVH_VOLUME_COL = "Volume"

# Output column names to add to CT_centroid.csv
OUT_MEAN_COL = "Mean Value"
OUT_MIN_COL = "Min Value"
OUT_MAX_COL = "Max Value"
OUT_VOLUME_COL = "Structure Volume"

# DRY RUN: if True, no CSV files are modified.
DRY_RUN = False
DRY_RUN_PREVIEW_CSV = "../data/dvh_merge_preview.csv"

# Requires `organ_rename_dict` in the notebook namespace
# (raw ROI name -> canonical organ name).


# ============================================================
# Step 0: build patient -> ID mapping from the Excel
# ============================================================

anon_df = pd.read_excel(ANON_MAPPING_XLSX, dtype=str).fillna("")
missing_cols = [c for c in (ANON_COL, ID_COL) if c not in anon_df.columns]
if missing_cols:
    raise ValueError(
        f"Excel missing required column(s) {missing_cols}. "
        f"Available columns: {list(anon_df.columns)}"
    )

anon_to_id = {}
for _, row in anon_df.iterrows():
    anon = str(row[ANON_COL]).strip()
    pid = str(row[ID_COL]).strip()
    if not anon or not pid:
        continue
    anon_to_id[anon] = pid


# ============================================================
# Step 0b: load tumor ROI selection CSV (per-patient GTV-P / GTV-N raw names)
# ============================================================

selection_df = pd.read_csv(TUMOR_ROI_SELECTION_CSV, dtype=str).fillna("")
selection_lookup = {}
for _, row in selection_df.iterrows():
    pid = str(row["patient_id"]).strip()
    gtv_p = row["GTV_p"].strip()
    gtv_n = row["GTV_n"].strip()
    if not pid:
        continue
    selection_lookup[pid] = {
        "gtv_p": gtv_p if gtv_p else None,
        "gtv_n": gtv_n if gtv_n else None,
    }


# ============================================================
# Step 1: load all DVH csv files
# ============================================================

dvh_files = sorted(glob.glob(os.path.join(DVH_DIR, "*.csv")))
if not dvh_files:
    raise FileNotFoundError(f"No DVH csv files found in {DVH_DIR}")

print(f"Loading {len(dvh_files)} DVH csv file(s)...")
dvh_frames = []
for f in dvh_files:
    try:
        df = pd.read_csv(f, dtype={DVH_ID_COL: str})
    except Exception as e:
        print(f"  [WARN] Could not read {f}: {e}")
        continue
    dvh_frames.append(df)

dvh_all = pd.concat(dvh_frames, ignore_index=True)

required_dvh_cols = [
    DVH_ID_COL, DVH_ROI_COL,
    DVH_MEAN_COL, DVH_MIN_COL, DVH_MAX_COL, DVH_VOLUME_COL,
]
missing_dvh = [c for c in required_dvh_cols if c not in dvh_all.columns]
if missing_dvh:
    raise ValueError(
        f"DVH csv missing required column(s) {missing_dvh}. "
        f"Available columns: {list(dvh_all.columns)}"
    )

dvh_all[DVH_ID_COL] = dvh_all[DVH_ID_COL].astype(str).str.strip()
dvh_all[DVH_ROI_COL] = dvh_all[DVH_ROI_COL].astype(str).str.strip()

dvh_by_id = {pid: g.reset_index(drop=True) for pid, g in dvh_all.groupby(DVH_ID_COL)}
print(f"DVH rows loaded: {len(dvh_all)}  |  unique IDs: {len(dvh_by_id)}")


# ============================================================
# Step 2: build lookup tables
#   - canonical organ name -> set of raw aliases (for DVH lookup)
#   - case-insensitive raw -> canonical (for the filter step)
# ============================================================

canonical_to_raw = defaultdict(set)
for raw_name, canonical in organ_rename_dict.items():
    canonical_to_raw[canonical].add(raw_name)

_RAW_TO_CANONICAL_LC = {
    str(k).strip().lower(): v for k, v in organ_rename_dict.items()
}
_ORGAN_LIST_LC = {str(o).strip().lower() for o in Const.organ_list}


def lookup_dvh_row(patient_dvh_df, centroid_roi_name):
    """
    Match logic:
      1. Try the ROI name as-is.
      2. If it's a canonical organ, also try each raw alias from organ_rename_dict.
    Returns the first matching row (Series) or None.
    """
    candidates = [centroid_roi_name]
    if centroid_roi_name in canonical_to_raw:
        candidates.extend(canonical_to_raw[centroid_roi_name])

    seen = set()
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        match = patient_dvh_df[patient_dvh_df[DVH_ROI_COL] == cand]
        if not match.empty:
            return match.iloc[0]
    return None


def is_kept_roi(roi_name, gtv_p_name, gtv_n_name):
    """Keep ROI if it maps to (or is) an organ in Const.organ_list,
    or equals this patient's GTV-P / GTV-N raw name."""
    key = str(roi_name).strip().lower()
    if key in _ORGAN_LIST_LC:
        return True
    canonical = _RAW_TO_CANONICAL_LC.get(key)
    if canonical is not None and str(canonical).strip().lower() in _ORGAN_LIST_LC:
        return True
    if gtv_p_name and roi_name == gtv_p_name:
        return True
    if gtv_n_name and roi_name == gtv_n_name:
        return True
    return False


# ============================================================
# Step 3: walk patient folders, merge DVH into CT_centroid,
#         then filter CT_centroid and CT_distances to
#         Const.organ_list + GTV-P + GTV-N (no folders are deleted).
# ============================================================

if not os.path.isdir(ROOT_DIR):
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

patient_folders = sorted(
    name for name in os.listdir(ROOT_DIR)
    if os.path.isdir(os.path.join(ROOT_DIR, name))
)

print()
print("DVH merge + filter to Const.organ_list ∪ {GTV-P, GTV-N}")
print(f"DRY_RUN = {DRY_RUN}")
print("=" * 100)

preview_rows = []
patients_no_id = []
patients_no_dvh = []
patients_missing_centroid = []
patients_with_unmatched = []
pending_updates = []  # list of (path, df) tuples

total_rois = 0
matched_rois = 0
total_kept_centroid = 0
total_kept_distance = 0
unmatched_examples = defaultdict(list)

for patient in patient_folders:
    folder = os.path.join(ROOT_DIR, patient)
    cen_path = os.path.join(folder, CENTROID_FILE)
    dis_path = os.path.join(folder, DISTANCE_FILE)

    if not os.path.isfile(cen_path):
        patients_missing_centroid.append(patient)
        print(f"{patient:30s} | [SKIP] no {CENTROID_FILE}")
        continue

    if patient not in anon_to_id:
        patients_no_id.append(patient)
        print(f"{patient:30s} | [SKIP] anonymized name not in Excel mapping")
        continue

    pid = anon_to_id[patient]
    patient_dvh = dvh_by_id.get(pid)
    has_dvh = patient_dvh is not None and not patient_dvh.empty
    if not has_dvh:
        patients_no_dvh.append(patient)

    sel = selection_lookup.get(patient, {"gtv_p": None, "gtv_n": None})
    gtv_p_name = sel.get("gtv_p")
    gtv_n_name = sel.get("gtv_n")

    df = pd.read_csv(cen_path)
    if CENTROID_ROI_COL not in df.columns:
        print(f"{patient:30s} | [WARN] '{CENTROID_ROI_COL}' column missing — skipping")
        continue

    # --- DVH match ---
    means, mins, maxs, volumes = [], [], [], []
    n_match = 0
    n_unmatch = 0
    for roi in df[CENTROID_ROI_COL].astype(str):
        total_rois += 1
        row = lookup_dvh_row(patient_dvh, roi) if has_dvh else None
        if row is None:
            means.append(pd.NA)
            mins.append(pd.NA)
            maxs.append(pd.NA)
            volumes.append(pd.NA)
            unmatched_examples[patient].append(roi)
            n_unmatch += 1
            if DRY_RUN:
                preview_rows.append({
                    "patient": patient, "ID": pid, "ROI": roi,
                    "matched": False,
                    OUT_MEAN_COL: None, OUT_MIN_COL: None, OUT_MAX_COL: None,
                    OUT_VOLUME_COL: None,
                })
            continue
        n_match += 1
        matched_rois += 1
        means.append(row[DVH_MEAN_COL])
        mins.append(row[DVH_MIN_COL])
        maxs.append(row[DVH_MAX_COL])
        volumes.append(row[DVH_VOLUME_COL])
        if DRY_RUN:
            preview_rows.append({
                "patient": patient, "ID": pid, "ROI": roi,
                "matched": True,
                OUT_MEAN_COL: row[DVH_MEAN_COL],
                OUT_MIN_COL: row[DVH_MIN_COL],
                OUT_MAX_COL: row[DVH_MAX_COL],
                OUT_VOLUME_COL: row[DVH_VOLUME_COL],
            })

    df[OUT_MEAN_COL] = means
    df[OUT_MIN_COL] = mins
    df[OUT_MAX_COL] = maxs
    df[OUT_VOLUME_COL] = volumes

    if n_unmatch > 0:
        patients_with_unmatched.append(patient)

    # --- Filter CT_centroid ---
    n_total_cen = len(df)
    cen_keep_mask = df[CENTROID_ROI_COL].astype(str).apply(
        lambda r: is_kept_roi(r, gtv_p_name, gtv_n_name)
    )
    cen_filtered = df[cen_keep_mask].reset_index(drop=True)
    n_kept_cen = len(cen_filtered)
    total_kept_centroid += n_kept_cen
    pending_updates.append((cen_path, cen_filtered))

    # --- Filter CT_distances ---
    n_kept_dis = None
    n_total_dis = None
    if os.path.isfile(dis_path):
        dis_df = pd.read_csv(dis_path)
        missing_dis_cols = [
            c for c in (DISTANCE_REF_COL, DISTANCE_TGT_COL) if c not in dis_df.columns
        ]
        if missing_dis_cols:
            print(f"{patient:30s} | [WARN] distances missing column(s) {missing_dis_cols}")
        else:
            n_total_dis = len(dis_df)
            ref_keep = dis_df[DISTANCE_REF_COL].astype(str).apply(
                lambda r: is_kept_roi(r, gtv_p_name, gtv_n_name)
            )
            tgt_keep = dis_df[DISTANCE_TGT_COL].astype(str).apply(
                lambda r: is_kept_roi(r, gtv_p_name, gtv_n_name)
            )
            dis_filtered = dis_df[ref_keep & tgt_keep].reset_index(drop=True)
            n_kept_dis = len(dis_filtered)
            total_kept_distance += n_kept_dis
            pending_updates.append((dis_path, dis_filtered))

    dis_str = (
        f"distances kept: {n_kept_dis}/{n_total_dis}"
        if n_kept_dis is not None else "distances: --"
    )
    print(
        f"{patient:30s} | ID={pid:15s} | "
        f"ROIs matched: {n_match}/{n_total_cen} (unmatched {n_unmatch}) | "
        f"centroid kept: {n_kept_cen}/{n_total_cen} | {dis_str}"
    )


# ============================================================
# Step 4: apply changes (or stage preview). No folders are deleted.
# ============================================================

print()
print("Summary")
print("-" * 100)
print(f"Patient folders scanned       : {len(patient_folders)}")
print(f"Missing {CENTROID_FILE:20s}  : {len(patients_missing_centroid)}")
print(f"No anon->ID in Excel          : {len(patients_no_id)}")
print(f"ID has no DVH rows            : {len(patients_no_dvh)}")
print(f"Patients with unmatched ROIs  : {len(patients_with_unmatched)}")
print(f"Total ROIs across patients    : {total_rois}")
print(f"Matched ROIs                  : {matched_rois}")
print(f"Unmatched ROIs                : {total_rois - matched_rois}")
print(f"Total centroid rows kept      : {total_kept_centroid}")
print(f"Total distance rows kept      : {total_kept_distance}")

if patients_no_id:
    print()
    print("Anonymized names not in Excel mapping (skipped, NOT deleted):")
    for p in patients_no_id:
        print(f"  - {p}")

if patients_no_dvh:
    print()
    print("Patients with no matching DVH rows (DVH columns are NaN, NOT deleted):")
    for p in patients_no_dvh:
        print(f"  - {p}  (ID={anon_to_id.get(p, '?')})")

if unmatched_examples:
    print()
    print("Unmatched ROI examples (up to 5 per patient):")
    for p, rois in list(unmatched_examples.items())[:50]:
        sample = ", ".join(rois[:5])
        more = f" (+{len(rois) - 5} more)" if len(rois) > 5 else ""
        print(f"  - {p:30s}: {sample}{more}")
    if len(unmatched_examples) > 50:
        print(f"  ... and {len(unmatched_examples) - 50} more patients with unmatched ROIs")

if DRY_RUN:
    if preview_rows:
        preview_df = pd.DataFrame(preview_rows)
        os.makedirs(os.path.dirname(DRY_RUN_PREVIEW_CSV) or ".", exist_ok=True)
        preview_df.to_csv(DRY_RUN_PREVIEW_CSV, index=False)
        print()
        print(f"[DRY RUN] No CSV files were modified.")
        print(f"[DRY RUN] Preview written to: {os.path.abspath(DRY_RUN_PREVIEW_CSV)}")
        print(f"[DRY RUN] Preview rows      : {len(preview_df)}")
        print()
        print("Preview head:")
        print(preview_df.head(15).to_string(index=False))
    else:
        print()
        print("[DRY RUN] No preview rows produced.")
else:
    for path, df_out in pending_updates:
        df_out.to_csv(path, index=False)
    print()
    print(f"Updated CSV file(s): {len(pending_updates)}")


Loading 7 DVH csv file(s)...
DVH rows loaded: 40747  |  unique IDs: 319

DVH merge + filter to Const.organ_list ∪ {GTV-P, GTV-N}
DRY_RUN = False
101417584648                   | ID=STIEFEL_364     | ROIs matched: 75/75 (unmatched 0) | centroid kept: 44/75 | distances kept: 1378/7626
102209659698                   | ID=STIEFEL_1042    | ROIs matched: 139/140 (unmatched 1) | centroid kept: 59/140 | distances kept: 1711/9730
102231587299                   | ID=STIEFEL_1298    | ROIs matched: 132/172 (unmatched 40) | centroid kept: 56/172 | distances kept: 1540/15051
102934140275                   | ID=STIEFEL_567     | ROIs matched: 119/122 (unmatched 3) | centroid kept: 54/122 | distances kept: 1431/7381
103512138945                   | ID=STIEFEL_1897    | ROIs matched: 126/127 (unmatched 1) | centroid kept: 55/127 | distances kept: 1485/8001
103895810595                   | ID=STIEFEL_2062    | ROIs matched: 114/124 (unmatched 10) | centroid kept: 54/124 | distances kept: 1431/7875
105

In [9]:
import os
import pandas as pd


# ============================================================
# Configuration
# ============================================================

ROOT_DIR = "../data/CAMPRT_Centroids_20260421_new"
CENTROID_FILE = "CT_centroid.csv"
DISTANCE_FILE = "CT_distances.csv"
TUMOR_ROI_SELECTION_CSV = "../data/tumor_roi_selection_20260421_new.csv"

# Column names in the CSV files
CENTROID_ROI_COL = "ROI"
DISTANCE_REF_COL = "Reference ROI"
DISTANCE_TGT_COL = "Target ROI"

# Canonical GTV names to use
CANONICAL_GTV_P = "GTV-P"
CANONICAL_GTV_N = "GTV-N"

# DRY RUN: if True, no files are modified. Preview is printed to console.
DRY_RUN = True


# ============================================================
# Step 0: load tumor ROI selection CSV to get original GTV names
# ============================================================

selection_df = pd.read_csv(TUMOR_ROI_SELECTION_CSV, dtype=str).fillna("")
# patient_id -> {"gtv_p": ..., "gtv_n": ... or None}
selection_lookup = {}
for _, row in selection_df.iterrows():
    pid = str(row["patient_id"]).strip()
    gtv_p = row["GTV_p"].strip()
    gtv_n = row["GTV_n"].strip()
    if not gtv_p:
        continue
    selection_lookup[pid] = {
        "gtv_p": gtv_p,
        "gtv_n": gtv_n if gtv_n else None,
    }


# ============================================================
# Step 1: rename GTV ROIs in each patient's CT_centroid and CT_distances
# ============================================================

if not os.path.isdir(ROOT_DIR):
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

patient_folders = sorted(
    name for name in os.listdir(ROOT_DIR)
    if os.path.isdir(os.path.join(ROOT_DIR, name))
)

print("Renaming GTV ROIs to canonical names")
print(f"DRY_RUN = {DRY_RUN}")
print("=" * 100)

processed = 0
patients_not_in_selection = []
patients_missing_centroid = []
patients_missing_distance = []

for patient in patient_folders:
    folder = os.path.join(ROOT_DIR, patient)
    
    if patient not in selection_lookup:
        patients_not_in_selection.append(patient)
        print(f"{patient:30s} | [SKIP] not in selection CSV")
        continue
    
    sel = selection_lookup[patient]
    gtv_p_original = sel["gtv_p"]
    gtv_n_original = sel["gtv_n"]
    
    # Build rename mapping: original name -> canonical name
    rename_map = {gtv_p_original: CANONICAL_GTV_P}
    if gtv_n_original is not None:
        rename_map[gtv_n_original] = CANONICAL_GTV_N
    
    cen_path = os.path.join(folder, CENTROID_FILE)
    dis_path = os.path.join(folder, DISTANCE_FILE)
    
    cen_renamed = 0
    dis_renamed = 0
    
    # Process CT_centroid.csv
    if os.path.isfile(cen_path):
        df = pd.read_csv(cen_path)
        if CENTROID_ROI_COL in df.columns:
            before = df[CENTROID_ROI_COL].tolist()
            df[CENTROID_ROI_COL] = df[CENTROID_ROI_COL].astype(str).map(
                lambda x: rename_map.get(x, x)
            )
            after = df[CENTROID_ROI_COL].tolist()
            cen_renamed = sum(1 for b, a in zip(before, after) if b != a)
            
            if not DRY_RUN:
                df.to_csv(cen_path, index=False)
        else:
            print(f"{patient:30s} | [WARN] '{CENTROID_ROI_COL}' column missing in centroid")
    else:
        patients_missing_centroid.append(patient)
    
    # Process CT_distances.csv
    if os.path.isfile(dis_path):
        df = pd.read_csv(dis_path)
        missing_cols = [c for c in (DISTANCE_REF_COL, DISTANCE_TGT_COL) if c not in df.columns]
        if not missing_cols:
            before_ref = df[DISTANCE_REF_COL].tolist()
            before_tgt = df[DISTANCE_TGT_COL].tolist()
            
            df[DISTANCE_REF_COL] = df[DISTANCE_REF_COL].astype(str).map(
                lambda x: rename_map.get(x, x)
            )
            df[DISTANCE_TGT_COL] = df[DISTANCE_TGT_COL].astype(str).map(
                lambda x: rename_map.get(x, x)
            )
            
            after_ref = df[DISTANCE_REF_COL].tolist()
            after_tgt = df[DISTANCE_TGT_COL].tolist()
            dis_renamed = (
                sum(1 for b, a in zip(before_ref, after_ref) if b != a) +
                sum(1 for b, a in zip(before_tgt, after_tgt) if b != a)
            )
            
            if not DRY_RUN:
                df.to_csv(dis_path, index=False)
        else:
            print(f"{patient:30s} | [WARN] Missing columns {missing_cols} in distances")
    else:
        patients_missing_distance.append(patient)
    
    gtv_n_str = f"{repr(gtv_n_original)} -> {CANONICAL_GTV_N}" if gtv_n_original else "None"
    print(
        f"{patient:30s} | "
        f"GTV_p: {repr(gtv_p_original)} -> {CANONICAL_GTV_P}  |  "
        f"GTV_n: {gtv_n_str}  |  "
        f"centroid: {cen_renamed} renamed  |  distances: {dis_renamed} renamed"
    )
    processed += 1


# ============================================================
# Summary
# ============================================================

print()
print("Summary")
print("-" * 100)
print(f"Patient folders scanned       : {len(patient_folders)}")
print(f"Patients processed            : {processed}")
print(f"Not in selection CSV          : {len(patients_not_in_selection)}")
print(f"Missing {CENTROID_FILE:20s}  : {len(patients_missing_centroid)}")
print(f"Missing {DISTANCE_FILE:20s}  : {len(patients_missing_distance)}")

if patients_not_in_selection:
    print()
    print("Patients not in selection CSV (skipped):")
    for p in patients_not_in_selection:
        print(f"  - {p}")

if patients_missing_centroid:
    print()
    print(f"Patients missing {CENTROID_FILE}:")
    for p in patients_missing_centroid:
        print(f"  - {p}")

if patients_missing_distance:
    print()
    print(f"Patients missing {DISTANCE_FILE}:")
    for p in patients_missing_distance:
        print(f"  - {p}")

if DRY_RUN:
    print()
    print("[DRY RUN] No files were modified.")
    print(f"[DRY RUN] Set DRY_RUN = False to apply changes.")
else:
    print()
    print(f"Renamed GTV ROIs in {processed} patient folder(s).")

Renaming GTV ROIs to canonical names
DRY_RUN = False
101417584648                   | [SKIP] not in selection CSV
105559195097                   | [SKIP] not in selection CSV
105852012614                   | GTV_p: 'GTV' -> GTV-P  |  GTV_n: None  |  centroid: 1 renamed  |  distances: 131 renamed
108372343816                   | GTV_p: 'GTV' -> GTV-P  |  GTV_n: 'GTV_Node' -> GTV-N  |  centroid: 2 renamed  |  distances: 254 renamed
110141315107                   | [SKIP] not in selection CSV
111661340239                   | GTV_p: 'GTV' -> GTV-P  |  GTV_n: None  |  centroid: 1 renamed  |  distances: 123 renamed
111691250024                   | GTV_p: 'GTV' -> GTV-P  |  GTV_n: 'GTV_Node_1' -> GTV-N  |  centroid: 2 renamed  |  distances: 268 renamed
111851899440                   | [SKIP] not in selection CSV
113809646383                   | [SKIP] not in selection CSV
116611465920                   | GTV_p: 'GTV' -> GTV-P  |  GTV_n: None  |  centroid: 1 renamed  |  distances: 108 renamed
